# 🔍 VisionDecode — Final Model

### Distorted Visual Sequence Pattern Recognition · CIG AI/ML Challenge (PS-3)

Modern systems often need to read text from images that are **noisy, blurred, occluded, overlapping, or warped**, where ordinary OCR breaks down. This challenge asks us to build a deep-learning model that recognises and reconstructs a **fixed-length character code** hidden inside a distorted grayscale image.

| | |
|---|---|
| **Task** | distorted image → ordered 6-character code |
| **Charset** | `0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ` (36 classes) |
| **Code length** | 6 characters (fixed) |
| **Distortions** | background noise · overlapping symbols · blur · shape deformation · occlusion · irregular spacing |
| **Metric** | **Character Error Rate (CER)** — Levenshtein-based, *lower is better* |

#### 🧭 Approach in this notebook

Because the code length is **fixed at 6**, we frame the problem as **6 parallel 36-way classification heads** (one per character position) instead of CTC decoding. The notebook flows top-to-bottom:

1. **Setup & imports** — device, seeds.
2. **Shared helpers** — charset encode/decode, a from-scratch **CER (Levenshtein)** metric, the `CaptchaDataset`, and seeded train/val dataloaders.
3. **Training pipeline** — a reusable `train_classifier` loop: per-position cross-entropy loss, `Adam`, `ReduceLROnPlateau`, best-checkpoint saving on validation CER.
4. **Model definitions** — three **CRNN** classifiers (custom CNN backbone → recurrent decoder) of differing capacity and recurrence.
5. **Ensemble** — a **weighted position-wise softmax soft-vote** that blends the three models.
6. **Submission** — the same soft-vote over the 5,000 test images, written in the required `image,prediction` format.

> 🏆 **Result (this run):** the weighted ensemble reaches **validation CER ≈ 0.00267** and **≈ 98.85 % full-code accuracy** — beating every individual member.

> 💡 This notebook is **fully self-contained**: it redefines every model class and helper, so it runs end-to-end independently of `prototype_model.ipynb`.

## 1 · Setup & Imports

Load PyTorch, torchvision and the usual data libraries, then pick the compute device and **fix the random seed (42)** so the train/val split and weight initialisation are reproducible across runs.

In [21]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
print('Using device:', device)

Using device: cuda


## 2 · Shared Helpers — charset, CER, dataset & loaders

Everything the models share lives here, so each piece is defined exactly once:

- **Charset & encoding** — `CHARS` (36 symbols) with `char_to_idx` / `idx_to_char` maps. `encode` turns a 6-char code into a `LongTensor` of class indices; `decode` reverses it.
- **`levenshtein` + `cer`** — the competition metric, implemented from scratch. `levenshtein` computes edit distance with an O(*m*) rolling DP row; `cer` aggregates total edits ÷ total reference characters across a batch (lower is better).
- **`CaptchaDataset`** — reads `train-labels.csv`, **filters to clean 6-char codes** whose characters are all in the charset, opens each image as RGB, and applies the transform. Returns `(image_tensor, label_indices)`.
- **`make_loaders`** — builds the dataset and makes a **seeded 90/10 train/val split** (`random_split` with a fixed generator) so every model and the ensemble see the *same* validation samples in the *same* order — essential for the soft-vote to stay label-aligned.
- **`imagenet_tf`** — resize to `100×200`, to-tensor, and ImageNet mean/std normalisation (the backbones expect 3-channel ImageNet-style input).

> ⚙️ Dataloaders use `num_workers=0` to avoid Jupyter forkserver `BrokenPipeError` on Windows/Python 3.14.

## Shared helpers (charset, CER, dataset, loaders)

In [2]:
CHARS = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
NUM_CLASSES = len(CHARS)
CODE_LEN = 6

char_to_idx = {c: i for i, c in enumerate(CHARS)}
idx_to_char = {i: c for i, c in enumerate(CHARS)}

def encode(text):
    return torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

def decode(indices):
    return ''.join(idx_to_char[int(i)] for i in indices)

def levenshtein(a, b):
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, m + 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
            prev = cur
    return dp[m]

def cer(preds, targets):
    edits = sum(levenshtein(p, t) for p, t in zip(preds, targets))
    chars = sum(len(t) for t in targets)
    return edits / max(chars, 1)

In [4]:
DATA_DIR = '../data'

class CaptchaDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        df = pd.read_csv(csv_file)
        df = df[df['text'].astype(str).str.len() == CODE_LEN].reset_index(drop=True)
        df = df[df['text'].apply(lambda t: all(c in char_to_idx for c in str(t)))].reset_index(drop=True)
        self.df,self.img_dir,self.transform = df, img_dir, transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir,row['image'])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img,encode(row['text'])

def make_loaders(transform, batch_size=64, val_frac=0.1, seed=42):
    full = CaptchaDataset(os.path.join(DATA_DIR, 'train-labels.csv'),
                          os.path.join(DATA_DIR, 'train_images'), transform)
    val_size = int(val_frac * len(full))
    train, val = random_split(full, [len(full) - val_size, val_size],
                              generator=torch.Generator().manual_seed(seed))
    tl = DataLoader(train,batch_size=batch_size,shuffle=True,num_workers=0)
    vl = DataLoader(val,batch_size=128,shuffle=False,num_workers=0)
    return tl, vl


imagenet_tf = transforms.Compose([
    transforms.Resize((100, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

## 3 · Training Pipeline

A single reusable trainer drives all three models so the comparison is fair.

- **`cls_loss`** — sums **cross-entropy over the 6 positions** independently. Each position is a 36-way classification, so the model learns one softmax per character slot.
- **`evaluate_cls`** — runs the model in eval mode and reports two numbers: the batch **CER** (the competition metric) and **full-code accuracy** (fraction of images where *all 6* characters are correct — a strictly harder, more honest score).
- **`train_classifier`** — `Adam` optimiser, **`ReduceLROnPlateau`** (halves the LR after 2 epochs without CER improvement), and **best-checkpoint saving**: the `.pth` is overwritten only when validation CER hits a new low, so the saved weights are always the best seen — not the last epoch.

> 📌 Models are selected on **validation CER**, the exact quantity the leaderboard scores, rather than on training loss.

In [22]:
def decode_logits(logits):
    idx = logits.argmax(dim=2).cpu()
    return [decode(row) for row in idx]

@torch.no_grad()
def evaluate_cls(model, loader):
    model.eval()
    preds, trues, seq_ok, seq_n = [], [], 0, 0
    for images, labels in loader:
        out = model(images.to(device))
        p = out.argmax(dim=2).cpu()
        for pi, ti in zip(p, labels):
            preds.append(decode(pi)); trues.append(decode(ti))
        seq_ok += (p == labels).all(dim=1).sum().item()
        seq_n  += labels.size(0)
    return cer(preds,trues), seq_ok/max(seq_n, 1)

def cls_loss(logits, labels):
    return sum(F.cross_entropy(logits[:,p,:], labels[:,p]) for p in range(CODE_LEN))

def train_classifier(model, train_loader, val_loader, epochs=15, lr=1e-3, ckpt=None):
    model = model.to(device)
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)
    best = float('inf')
    if ckpt:
        os.makedirs(os.path.dirname(ckpt), exist_ok=True)
    for epoch in range(1, epochs + 1):
        model.train(); running = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            loss = cls_loss(model(images), labels)
            loss.backward(); opt.step()
            running += loss.item() * images.size(0)
        tr = running / len(train_loader.dataset)
        vcer, sacc = evaluate_cls(model, val_loader); sched.step(vcer)
        print(f'Epoch {epoch:2d} | loss {tr:.4f} | val CER {vcer:.4f} | full-code acc {sacc:.4f}')
        if ckpt and vcer < best:
            best = vcer; torch.save(model.state_dict(), ckpt)
            print(f'  ↳ best CER {best:.4f} saved -> {ckpt}')
    return best



## 4 · Model Definitions — three CRNNs

All three are **CRNNs**: a **custom from-scratch CNN backbone** extracts visual features, an `AdaptiveAvgPool2d((1, 6))` collapses them into a length-6 sequence (one step per character slot), a **bidirectional recurrent decoder** models left/right context between neighbouring characters, and a shared `Linear` head produces the `(B, 6, 36)` logit grid.

The three differ in backbone depth and recurrence, giving the ensemble **diverse error patterns** to average over:

| Member | Class | Backbone (final width) | Recurrent decoder |
|--------|-------|------------------------|-------------------|
| **Model 1** | `CRNN_CustomModel_light` | 5-block CNN → **512** | 2-layer **BiGRU** (dropout 0.3) |
| **Model 2** | `CRNN_CustomModel` | deeper CNN → **1024** (1×1 expansion) | 4-layer **BiGRU** (dropout 0.4) |
| **Model 3** | `CRNN_CustomModel_light_lstm` | 5-block CNN → **512** | 4-layer **BiLSTM** (dropout 0.35) |

**Shared forward pass:** `backbone → pool((1, 6)) → squeeze → permute to [B, 6, C] → RNN → Linear → [B, 6, 36]`.

> 🔁 GRU vs. LSTM and shallow vs. deep recurrence make the members **decorrelated**, which is exactly what makes the soft-vote ensemble pay off.

In [6]:
class CRNN_CustomModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=128):
        super().__init__()
        self.code_len = code_len
        self.hidden = hidden
        
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 1024, kernel_size=1, bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True),
        )
        
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))        
        self.gru = nn.GRU(1024, hidden, num_layers=4, batch_first=True,
                          bidirectional=True, dropout=0.4)
        
        self.fc = nn.Linear(hidden * 2, num_classes)
        
    
    def forward(self, x):
        features = self.backbone(x)  
        
        features = self.pool(features)  # [B, 1024, 1, code_len]
        features = features.squeeze(2)   # [B, 1024, code_len]
        
        features = features.permute(0, 2, 1)  # [B, code_len, 1024]
        
        seq, _ = self.gru(features)  # [B, code_len, hidden*2]
        output = self.fc(seq)  # [B, code_len, num_classes]
        
        return output

class CRNN_CustomModel_light(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=128):
        super().__init__()
        self.code_len = code_len
        self.hidden = hidden
        
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))        
        self.gru = nn.GRU(512, hidden, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=0.3)
        
        self.fc = nn.Linear(hidden * 2, num_classes)
            
    def forward(self, x):
        features = self.backbone(x)  
        
        features = self.pool(features)  # [B, 1024, 1, code_len]
        features = features.squeeze(2)   # [B, 1024, code_len]
        
        features = features.permute(0, 2, 1)  # [B, code_len, 1024]
        
        seq, _ = self.gru(features)  # [B, code_len, hidden*2]
        output = self.fc(seq)  # [B, code_len, num_classes]
        
        return output
    
class CRNN_CustomModel_light_lstm(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=128):
        super().__init__()
        self.code_len = code_len
        self.hidden = hidden
        
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))        
        self.lstm = nn.LSTM(512, hidden, num_layers=4, batch_first=True,
                          bidirectional=True, dropout=0.35)
        
        self.fc = nn.Linear(hidden * 2, num_classes)
            
    def forward(self, x):
        features = self.backbone(x)  
        
        features = self.pool(features)  # [B, 1024, 1, code_len]
        features = features.squeeze(2)   # [B, 1024, code_len]
        
        features = features.permute(0, 2, 1)  # [B, code_len, 1024]
        
        seq, _ = self.lstm(features)  # [B, code_len, hidden*2]
        output = self.fc(seq)  # [B, code_len, num_classes]
        
        return output

In [7]:
model1 = CRNN_CustomModel_light().to(device)
model2 = CRNN_CustomModel().to(device)
model3 = CRNN_CustomModel_light_lstm().to(device)

print('Model1 params:', sum(p.numel() for p in model1.parameters() if p.requires_grad))
print('Model2 params:', sum(p.numel() for p in model2.parameters() if p.requires_grad))
print('Model3 params:', sum(p.numel() for p in model3.parameters() if p.requires_grad))

Model1 params: 2368324
Model2 params: 6241092
Model3 params: 3422020


### Trainable parameters

| Member | Class | Trainable params |
|--------|-------|-----------------:|
| Model 1 | `CRNN_CustomModel_light` | ~2.37 M |
| Model 2 | `CRNN_CustomModel` | ~6.24 M |
| Model 3 | `CRNN_CustomModel_light_lstm` | ~3.42 M |

All three are compact (≤ ~6 M params) and train comfortably on a single mid-range GPU.

In [8]:
train_loader, val_loader = make_loaders(imagenet_tf)

### Train the three members

Each model trains for up to 75 epochs with the shared `train_classifier`, saving its best-CER checkpoint to `../Artifacts/`. Best validation results from this run:

| Member | Checkpoint | Best val CER ↓ | Full-code acc ↑ |
|--------|------------|---------------:|----------------:|
| Model 1 — `CRNN_CustomModel_light` | `CRNN_CustomModel_light.pth` | **0.0051** | ~97.7 % |
| Model 2 — `CRNN_CustomModel` | `CRNN_CustomModel.pth` | 0.0128 | ~92.9 % |
| Model 3 — `CRNN_CustomModel_light_lstm` | `CRNN_CustomModel_light_lstm.pth` | 0.0068 | ~96.4 % |

> 💡 Already-trained checkpoints? Skip these three cells and jump straight to the **Ensemble** section, which reloads the weights from `../Artifacts/`.

In [ ]:
train_classifier(model2, train_loader, val_loader, epochs=75, lr=1e-3,
                 ckpt='../Artifacts/CRNN_CustomModel.pth')

Epoch  1 | loss 0.8791 | val CER 0.0661 | full-code acc 0.6648
  ↳ best CER 0.0661 saved -> ../Artifacts/CRNN_CustomModel.pth
Epoch  2 | loss 0.8664 | val CER 0.0642 | full-code acc 0.6688
  ↳ best CER 0.0642 saved -> ../Artifacts/CRNN_CustomModel.pth
Epoch  3 | loss 0.8103 | val CER 0.0874 | full-code acc 0.5783
Epoch  4 | loss 0.8350 | val CER 0.0555 | full-code acc 0.7134
  ↳ best CER 0.0555 saved -> ../Artifacts/CRNN_CustomModel.pth
Epoch  5 | loss 0.7673 | val CER 0.0432 | full-code acc 0.7729
  ↳ best CER 0.0432 saved -> ../Artifacts/CRNN_CustomModel.pth
Epoch  6 | loss 0.6545 | val CER 0.0519 | full-code acc 0.7294
Epoch  7 | loss 0.7121 | val CER 0.0790 | full-code acc 0.6033
Epoch  8 | loss 0.6235 | val CER 0.0410 | full-code acc 0.7834
  ↳ best CER 0.0410 saved -> ../Artifacts/CRNN_CustomModel.pth
Epoch  9 | loss 0.5837 | val CER 0.0428 | full-code acc 0.7789
Epoch 10 | loss 0.6561 | val CER 0.0622 | full-code acc 0.6813
Epoch 11 | loss 0.6068 | val CER 0.0369 | full-code acc

0.012756378189094548

In [ ]:
train_classifier(model3, train_loader, val_loader, epochs=75, lr=1e-3,
                 ckpt='../Artifacts/CRNN_CustomModel_light_lstm.pth')

Epoch  1 | loss 0.4712 | val CER 0.0418 | full-code acc 0.7779
  ↳ best CER 0.0418 saved -> ../Artifacts/CRNN_CustomModel_light_lstm.pth
Epoch  2 | loss 0.4276 | val CER 0.0502 | full-code acc 0.7419
Epoch  3 | loss 0.3938 | val CER 0.0400 | full-code acc 0.7889
  ↳ best CER 0.0400 saved -> ../Artifacts/CRNN_CustomModel_light_lstm.pth
Epoch  4 | loss 0.3454 | val CER 0.0394 | full-code acc 0.7899
  ↳ best CER 0.0394 saved -> ../Artifacts/CRNN_CustomModel_light_lstm.pth
Epoch  5 | loss 0.2960 | val CER 0.0266 | full-code acc 0.8569
  ↳ best CER 0.0266 saved -> ../Artifacts/CRNN_CustomModel_light_lstm.pth
Epoch  6 | loss 0.3163 | val CER 0.0201 | full-code acc 0.8924
  ↳ best CER 0.0201 saved -> ../Artifacts/CRNN_CustomModel_light_lstm.pth
Epoch  7 | loss 0.2811 | val CER 0.0231 | full-code acc 0.8734
Epoch  8 | loss 0.2285 | val CER 0.0259 | full-code acc 0.8619
Epoch  9 | loss 0.2405 | val CER 0.0307 | full-code acc 0.8359
Epoch 10 | loss 0.1028 | val CER 0.0115 | full-code acc 0.9395


0.006753376688344172

## 5 · Ensemble — weighted soft-vote

The three members are combined by **averaging their weighted softmax probabilities position-wise**, then taking the `argmax` per character slot:

```
prediction[pos] = argmax( 0.4·P₁ + 0.2·P₂ + 0.4·P₃ )[pos]      # softmax probs, per character position
```

The weights `[0.4, 0.2, 0.4]` down-weight Model 2 (the weakest member) and lean on the two strongest CRNNs. Soft-voting on **probabilities** (not hard labels) lets a confident model rescue a position where another is unsure.

**Why the labels stay aligned:** every member uses the same `imagenet_tf` transform and `make_loaders` re-uses the **fixed seed**, so each per-model validation loader iterates the *identical* samples in the *identical* order — the probability tensors line up row-for-row.

In [23]:
@torch.no_grad()
def ensemble_predict(models_with_transforms, loader_builder, weights=None):
    n = len(models_with_transforms)
    weights = weights or [1.0] * n
    loaders = [loader_builder(tf)[1] for _, tf in models_with_transforms]  
    for m, _ in models_with_transforms:
        m.eval().to(device)

    preds, trues = [], []
    iters = [iter(l) for l in loaders]
    while True:
        try:
            batches = [next(it) for it in iters]
        except StopIteration:
            break
        labels = batches[0][1]
        prob_sum = 0.0
        for w, (m, _), (images, _) in zip(weights, models_with_transforms, batches):
            prob_sum = prob_sum + w * F.softmax(m(images.to(device)), dim=2).cpu()
        idx = prob_sum.argmax(dim=2)
        for pi, ti in zip(idx, labels):
            preds.append(decode(pi)); trues.append(decode(ti))
    seq_acc = np.mean([p == t for p, t in zip(preds, trues)])
    return cer(preds,trues),seq_acc,preds


In [26]:
ART = '../Artifacts'

model1 = CRNN_CustomModel_light().to(device)
model1.load_state_dict(torch.load(f'{ART}/CRNN_CustomModel_light.pth', map_location=device))

model2 = CRNN_CustomModel().to(device)
model2.load_state_dict(torch.load(f'{ART}/CRNN_CustomModel.pth', map_location=device))

model3 = CRNN_CustomModel_light_lstm().to(device)
model3.load_state_dict(torch.load(f'{ART}/CRNN_CustomModel_light_lstm.pth', map_location=device))

combo = [(model1,imagenet_tf), (model2,imagenet_tf), (model3,imagenet_tf)]
WEIGHTS = [0.5, 0.1, 0.4]

e_cer, e_acc, _ = ensemble_predict(combo,make_loaders,weights=WEIGHTS)
print(f'Ensemble  | val CER {e_cer:.8f} | full-code acc {e_acc:.6f}')

Ensemble  | val CER 0.00258463 | full-code acc 0.989495


## 🏆 Final Ensemble — Results

### Validation performance (this run)

| Model | Val CER ↓ | Full-code Acc ↑ |
|-------|----------:|----------------:|
| Model 1 — `CRNN_CustomModel_light` (BiGRU-2) | 0.0051 | ~97.7 % |
| Model 2 — `CRNN_CustomModel` (BiGRU-4) | 0.0128 | ~92.9 % |
| Model 3 — `CRNN_CustomModel_light_lstm` (BiLSTM-4) | 0.0068 | ~96.4 % |
| **Weighted soft-vote (0.5 / 0.1 / 0.4)** | **0.00258** | **🟢 98.95 %** |

### Key takeaways
- ✅ **The ensemble beats every individual member** — CER drops to **0.00258** (~0.26 %) and full-code accuracy rises to **98.95 %**.
- ✅ **Diversity drives the gain** — mixing GRU/LSTM and shallow/deep recurrence decorrelates the members' mistakes, so the soft-vote corrects positions where any single model slips.
- ✅ **Lightweight & reproducible** — three compact CRNNs (≤ ~6 M params each) trained from scratch on a single GPU, with a fixed seed and best-CER checkpointing.

> ℹ️ Numbers reflect the **current weights `[0.5, 0.1, 0.4]`**. Re-weighting the blend (e.g. leaning harder on the lowest-CER member) shifts these — tune on the validation split before submitting.

## 6 · Generate the Final Submission CSV

Run the same weighted soft-vote over every image in `../data/test_images` and write the predictions in the required format.

- **`TestDataset`** — loads the unlabelled test PNGs (sorted) and returns `(image_tensor, filename)`.
- **`ensemble_submit`** — soft-votes the three models exactly as in validation, then writes a `image,prediction` CSV (one row per test image).

The output is saved as `../Result/submission_<name>_<enroll>.csv` — the file name encodes the participant per the challenge guidelines. The cell below then **re-sorts rows by the numeric test index** so the CSV reads `test-0, test-1, test-2, …` rather than lexicographic order.

> 📝 **Before submitting:** replace `<name>` / `<enroll>` in `out_path` with your own details (currently `AryanSharma_24113024`).

In [27]:
class TestDataset(Dataset):
    def __init__(self, img_dir, transform):
        self.paths = sorted(glob.glob(os.path.join(img_dir, '*.png')))
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        return self.transform(Image.open(p).convert('RGB')), os.path.basename(p)

def _test_collate(batch):
    imgs, names = zip(*batch)
    return torch.stack(imgs,0),list(names)

@torch.no_grad()
def ensemble_submit(models_with_transforms, test_dir, weights=None,
                    out_path='../submission_<name>_<enroll>.csv', batch_size=128):
    n = len(models_with_transforms)
    weights = weights or [1.0] * n
    loaders = [DataLoader(TestDataset(test_dir, tf), batch_size=batch_size,
                          shuffle=False, num_workers=0, collate_fn=_test_collate)
               for _, tf in models_with_transforms]
    for m, _ in models_with_transforms:
        m.eval().to(device)

    rows = []
    iters = [iter(l) for l in loaders]
    while True:
        try:
            batches = [next(it) for it in iters]
        except StopIteration:
            break
        names = batches[0][1]
        prob_sum = 0.0
        for w, (m, _), (images, _) in zip(weights, models_with_transforms, batches):
            prob_sum = prob_sum + w * F.softmax(m(images.to(device)), dim=2).cpu()
        idx = prob_sum.argmax(dim=2)
        for name, pi in zip(names, idx):
            rows.append((name, decode(pi)))

    sub = pd.DataFrame(rows, columns=['image', 'prediction'])
    sub.to_csv(out_path, index=False)
    print(f'Wrote {len(sub)} predictions -> {out_path}')
    return sub

submission = ensemble_submit(combo, os.path.join(DATA_DIR, 'test_images'),
                             weights=WEIGHTS,
                             out_path='../Result/submission_AryanSharma_24113024.csv')
submission.head(10)

Wrote 5000 predictions -> ../Result/submission_AryanSharma_24113024.csv


,image,prediction
0,test-0.png,QVTQ8A
1,test-1.png,7PSW9D
2,test-10.png,7DUP98
3,test-100.png,75Z4WT
4,test-1000.png,QAKZ7V
5,test-1001.png,R6MERY
6,test-1002.png,CHXX67
7,test-1003.png,9NV2WP
8,test-1004.png,F56TDZ
9,test-1005.png,FFTFRX


In [28]:
df = pd.read_csv('../Result/submission_AryanSharma_24113024.csv')
df["index"] = df['image'].str.split('-').str[1].str.split('.').str[0]
df["index"] = df["index"].astype(int)
df = df.sort_values('index').reset_index(drop=True)
df = df.drop(columns=['index'])
df.to_csv('../Result/submission_AryanSharma_24113024.csv', index=False)

## ✅ Summary

This notebook builds an end-to-end pipeline for **distorted 6-character code recognition**:

1. **Framing** — fixed-length codes → 6 parallel 36-way classification heads (no CTC needed).
2. **Three diverse CRNNs** — custom CNN backbones (512 / 1024 / 512 wide) paired with BiGRU/BiLSTM decoders of varying depth, each selected on best **validation CER**.
3. **Weighted soft-vote ensemble** — averaging position-wise softmax probabilities `[0.4, 0.2, 0.4]` yields **val CER ≈ 0.00267 / 98.85 % full-code accuracy**, beating every member.
4. **Reproducible submission** — the same blend over the 5,000 test images, written and index-sorted to `Result/submission_AryanSharma_24113024.csv`.

**Possible next steps:** grid-search the ensemble weights on the validation split, add test-time augmentation, or fold in a CTC head as a fourth, differently-biased member.